# Step-1: Preprocessing - Clean, Split(train, test)

In [18]:
import pandas as pd

# Load dataset and strip leading/trailing spaces from column names
df = pd.read_csv("./data.csv")
df.columns = df.columns.str.strip()

# Check shape (must be >= 500 rows and >= 12 features) and class balance
# Dataset has 6819 rows and 96 columns (95 features + 1 target 'Bankrupt?')
print("Dataset Shape:", df.shape)
print("Target Distribution:\n", df["Bankrupt?"].value_counts())

Dataset Shape: (6819, 96)
Target Distribution:
 Bankrupt?
0    6599
1     220
Name: count, dtype: int64


In [19]:
target_col = "Bankrupt?"

# Partition dataset by binary target: Bankrupt (1, 220 instances) and Non-Bankrupt (0, 6599 instances).
bankrupt_df = df[df[target_col] == 1]
non_bankrupt_df = df[df[target_col] == 0]

In [20]:
from sklearn.model_selection import train_test_split

# Split Bankrupt: 30% to train (66 instances), 70% to test (154 instances).
b_train, b_test = train_test_split(bankrupt_df, test_size=0.70, random_state=99)

# Split Non-Bankrupt: 70% to train (4619 instances), 30% to test (1980 instances).
nb_train, nb_test = train_test_split(non_bankrupt_df, test_size=0.30, random_state=99)

# Combine splits to produce train_data (4685 rows) and test_data (2134 rows).
train_df = pd.concat([b_train, nb_train]).sample(frac=1, random_state=99).reset_index(drop=True)
test_df = pd.concat([b_test, nb_test]).sample(frac=1, random_state=99).reset_index(drop=True)

In [21]:
# Export both datasets with features and target preserved for model training and Streamlit evaluation.
train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

# Step-2: Model Training

In [2]:
import os

os.makedirs('./model', exist_ok=True)

In [4]:
import pandas as pd

# Read the saved train dataset directly
train_df = pd.read_csv("./train_data.csv")

target_col = "Bankrupt?"
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Define the 5 mandatory classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=99),
    "Decision Tree": DecisionTreeClassifier(random_state=99),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=99)
}

In [10]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

# Fit and persist each pipeline
for name, clf in models.items():
    pipeline = Pipeline([
        ("var_thresh", VarianceThreshold(threshold=0.0)),
        ("scaler", StandardScaler()),
        ("classifier", clf)
    ])
    
    pipeline.fit(X_train, y_train)
    
    filename = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, os.path.join("model", filename))

/Users/thedeveloperswork/Workspace/GitHub/2026AJ05314/ml_assignment_2/.venv/lib/python3.15/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
